# Useful Notebook: Interactively View Alternative Decision Thresholds for a Trained Model
**This notebook will allow users to interactively examine different decision thresholds for a target model, and see how this new threshold impacts confusion matrix metrics (i.e. TP, TN, FP, FN).**

*This notebook is designed to run after having run STREAMLINE (at least phases 1-6) and will use the files from a specific STREAMLINE experiment folder, as well as save new output files to that same folder.*

***
## Notebook Details
Allows users to interactively examine different decision thresholds (rather than the standard .5 probability of case) for a target model and see how new thresholds impact model performance metrics.

This notebook is designed to be run on a single model (i.e. specific dataset, CV, and algorithm).

***
## Notebook Run Parameters
* This notbook has been set up to run 'as-is' on the experiment folder generated when running the demo of STREAMLINE in any mode (if no run parameters were changed).
* If you have run STREAMLINE on different target data or saved the experiment to some other folder outside of STREAMLINE, you need to edit `experiment_path` below to point to the respective experiment folder.

In [ ]:
experiment_path = "/Users/harshbandhey/Local/Cedars/Urbslab/STREAMLINEv3New/test/out_full_pipeline/DemoExp" # path the target experiment folder 
targetDataName = 'hcc_survival' # specify a specific dataset
algorithm = 'Decision Tree' # specify algorithm
cvCount = 0 #specify the CV number

***
## Housekeeping
### Import Packages

In [ ]:
%matplotlib notebook
from ipywidgets import *

import os
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import metrics
pd.options.display.float_format = "{:.4f}".format
sns.set(palette='rainbow', context='talk')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.svm import SVC
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from matplotlib.widgets import Slider

import pickle
import seaborn as sns
from sklearn import metrics

import warnings
warnings.filterwarnings('ignore')

# Jupyter Notebook Hack: This code ensures that the results of multiple commands within a given cell are all displayed, rather than just the last. 
#from IPython.core.interactiveshell import InteractiveShell
#InteractiveShell.ast_node_interactivity = "all"

### Automatically detect data folder names

In [ ]:
# Get dataset paths for all completed dataset analyses in experiment folder
experiment_name = experiment_path.split('/')[-1]
remove_list = {
    '.DS_Store', 'metadata.pickle', 'metadata.csv', 'algInfo.pickle',
    'DatasetComparisons', 'jobs', 'jobsCompleted', 'logs', 'KeyFileCopy', 'dask_logs',
    'reporting', 'reporting_replication', 'run_params.pickle', 'runtime',
    experiment_name + '_STREAMLINE_Report.pdf'
}

datasets = []
for d in sorted(os.listdir(experiment_path)):
    dpath = os.path.join(experiment_path, d)
    if d in remove_list or not os.path.isdir(dpath):
        continue
    has_exploratory = os.path.isdir(os.path.join(dpath, 'exploratory'))
    has_model_data = os.path.isdir(os.path.join(dpath, 'model_evaluation')) or os.path.isdir(os.path.join(dpath, 'models'))
    if has_exploratory and has_model_data:
        datasets.append(d)

print("Analyzed Datasets: " + str(datasets))

### Load other necessary parameters

In [ ]:
#Unpickle metadata from previous phase
file = open(experiment_path+'/'+"metadata.pickle", 'rb')
metadata = pickle.load(file)
file.close()
#Load variables specified earlier in the pipeline from metadata
outcome_label = metadata.get('Outcome Label', metadata.get('Class Label', 'Class'))
instance_label = metadata['Instance Label']
cv_partitions = int(metadata['CV Partitions'])
primary_metirc = metadata.get('Primary Metric', metadata.get('P6 Scoring Metric', metadata.get('P8 Scoring Metric', 'balanced_accuracy')))

#Unpickle algorithm information from previous phase
alg_info_path = os.path.join(experiment_path, "algInfo.pickle")
if os.path.exists(alg_info_path):
    with open(alg_info_path, "rb") as file:
        algInfo = pickle.load(file)
else:
    algInfo = {}

algorithms = []
abbrev = {}
colors = {}
for key, value in algInfo.items():
    if isinstance(value, (list, tuple)) and len(value) > 0 and bool(value[0]):
        algorithms.append(key)
        abbrev[key] = value[1] if len(value) > 1 else key
        colors[key] = value[2] if len(value) > 2 else None

# Fallback: infer algorithms from current output layout
if not algorithms:
    inferred = set()
    scan_datasets = [d for d in datasets if os.path.isdir(os.path.join(experiment_path, d))]
    for ds_name in scan_datasets:
        model_dir = os.path.join(experiment_path, ds_name, "models", "pickledModels")
        if os.path.isdir(model_dir):
            for fname in os.listdir(model_dir):
                if fname.endswith('.pickle') and '_' in fname:
                    inferred.add(fname.rsplit('_', 1)[0])
        metric_dir = os.path.join(experiment_path, ds_name, "model_evaluation", "metrics_by_cv")
        if os.path.isdir(metric_dir):
            for fname in os.listdir(metric_dir):
                if fname.endswith('.json') and '_CV_' in fname:
                    inferred.add(fname.split('_CV_')[0])
    for abr in sorted(inferred):
        algorithms.append(abr)
        abbrev[abr] = abr
        colors[abr] = None

palette = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]
for idx, key in enumerate(algorithms):
    if colors.get(key) is None:
        colors[key] = palette[idx % len(palette)]

print("Algorithms Ran: " + str(algorithms))

## Define Necessary Methods

In [ ]:
def lnsp(initial, max, num):
    diff = max - initial
    list1 = []
    for i in range(num):
        list1.append(initial + (i * diff/num))
    return list1


def get_fp_tp(y_test, proba, thresh):
    pred = []
    fp = 0
    tp = 0
    fn = 0
    tn = 0
    threshold = round(thresh, 2)
    for i in range(len(proba)):
        if proba[i] >= threshold:
            pred.append(1)
        elif proba[i] < threshold:
            pred.append(0)
    np.asarray(y_test)
    y_test = y_test.tolist()
    for i in range(len(y_test)):
        if y_test[i] == pred[i]:
            if y_test[i] == 1:
                tp += 1
            elif y_test[i] == 0:
                tn += 1
        elif y_test[i] != pred[i]:
            if pred[i]== 1:
                fp += 1
            elif pred[i] == 0:
                fn += 1
    return fp, tp, fn, tn

def get_fpr_tpr(y_test, proba):
    negatives = np.sum(y_test == 0)
    positives = np.sum(y_test == 1)
    columns = ['threshold', 'false_positive_rate', 'true_positive_rate']
    fptp = pd.DataFrame(columns=columns, dtype=np.number)
    thresholds = np.linspace(0, 1, 101)
    for i, threshold in enumerate(thresholds):
        fptp.loc[i, 'threshold'] = threshold
        false_positives, true_positives, fn, tn = get_fp_tp(y_test, proba, threshold)
        fptp.loc[i, 'false_positive_rate'] = false_positives / negatives
        fptp.loc[i, 'true_positive_rate'] = true_positives / positives
    fptp.head(15)
    return fptp

def get_tfpn(y_test, proba):
    columns = ['threshold', 'false_positives', 'true_positives', 'false_negatives',]
    tfpn = pd.DataFrame(columns=columns, dtype=np.number)
    thresholds = np.linspace(0, 1, 101)
    for i, threshold in enumerate(thresholds):
        tfpn.loc[i, 'threshold'] = round(threshold, 2)
        false_positives, true_positives, false_negatives, true_negatives = get_fp_tp(y_test, proba, round(threshold, 2))
        tfpn.loc[i, 'false_positives'] = false_positives
        tfpn.loc[i, 'true_positives'] = true_positives
        tfpn.loc[i, 'false_negatives'] = false_negatives
        tfpn.loc[i, 'true_negatives'] = true_negatives
    return tfpn

def cm_maker(y_test, proba, thresh):
    tfpn = get_tfpn(y_test, proba)
    tfpn_partial = tfpn.loc[tfpn['threshold'] == thresh]
    #print(np.array(tfpn_partial.true_positives))
    cm_part_1 = np.array(tfpn_partial.true_positives)
    cm_part_2 = np.array(tfpn_partial.false_negatives)
    cm_part_3 = np.array(tfpn_partial.false_positives)
    cm_part_4 = np.array(tfpn_partial.true_negatives)
    merge = np.concatenate((cm_part_1, cm_part_2))
    merge_2 = np.concatenate((cm_part_3, cm_part_4))
    cm_final = np.concatenate(([merge], [merge_2]))
    return cm_final

def graph_roc(y_test, proba, fig, ax, threshold, tpr, fpr, AUC):
    # ax = ax.flatten()
    auc=metrics.roc_auc_score(y_test, proba)
    # ax.plot(fpr, tpr, label="AUC=" + str(auc))
    ax.plot(fpr, tpr, label='AUC:' + str(AUC))
    ax.set(ylabel = ('True Positive Rate'),
    xlabel = ('False Positive Rate'))
    ax.legend()

def getdata(threshold, fprtpr):
    point = fprtpr.loc[fprtpr['threshold'] == threshold]
    fpr = np.array(point.false_positive_rate)
    tpr = np.array(point.true_positive_rate)
    return fpr, tpr

def graph_line(ax, threshold, fprtpr):
    fpr, tpr = getdata(threshold, fprtpr)
    vert_line = ax.axvline(x=fpr, color='gray', linestyle='--')
    hori_line = ax.axhline(y=tpr, color='gray', linestyle='--')

        # vert_line.remove(vert_line)
        # vert_line = ax.axvline(x=(threshold*0.2), color='gray', linestyle='--')
    return vert_line, hori_line

def plot_point(ax, threshold, fprtpr):
    fpr, tpr = getdata(threshold, fprtpr)
    ptplt = ax.plot(fpr, tpr, color='blue', marker='o', markersize=8)
    ax.legend()
    return ptplt

def get_cms(y_test, proba):
    thresholds = np.linspace(0, 1, 101)
    ls = []
    for i in thresholds:
        cm = cm_maker(y_test, proba, round(i, 2)).tolist()
        ls.append(cm)
    return ls

def get_auc(model, x_test, y_test):
    y_predict = model.predict(x_test)
    AUC = metrics.roc_auc_score(y_test, y_predict)
    print(AUC)


## Activate an Interactive Window to Explore Model Decision Thresholds

In [ ]:
global point, hline, vline, coordinate

full_path = experiment_path+'/'+targetDataName

# Resolve selected algorithm key
selected_algorithm = algorithm
if selected_algorithm not in abbrev:
    reverse_map = {v: k for k, v in abbrev.items()}
    if selected_algorithm in reverse_map:
        selected_algorithm = reverse_map[selected_algorithm]
if selected_algorithm not in abbrev:
    if not algorithms:
        raise ValueError("No algorithms were discovered in this experiment.")
    print("Requested algorithm not found. Using first available algorithm: " + str(algorithms[0]))
    selected_algorithm = algorithms[0]
if cvCount >= cv_partitions:
    raise ValueError("cvCount {} exceeds available CV partitions {}".format(cvCount, cv_partitions))

model_file = full_path+'/models/pickledModels/'+abbrev[selected_algorithm]+"_"+str(cvCount)+'.pickle'
with open(model_file, 'rb') as file:
    model = pickle.load(file)

#load testing data
test_file_path = full_path + '/CVDatasets/' + targetDataName + "_CV_" + str(cvCount) + "_Test.csv"
test = pd.read_csv(test_file_path)
if instance_label != 'None' and instance_label in test.columns:
    test = test.drop(instance_label, axis=1)
x_test = test.drop(outcome_label,axis=1).values
y_test = test[outcome_label].values

del test #memory cleanup

if not hasattr(model, 'predict_proba'):
    raise ValueError("Selected model does not support predict_proba.")
proba_all = model.predict_proba(x_test)
if len(proba_all.shape) != 2 or proba_all.shape[1] < 2:
    raise ValueError("DecisionThreshold_Interactive expects probability output.")

classes = list(getattr(model, 'classes_', range(proba_all.shape[1])))
if proba_all.shape[1] > 2 or len(np.unique(y_test)) > 2:
    positive_class = 1 if 1 in classes else classes[0]
    class_index = classes.index(positive_class)
    y_eval = (y_test == positive_class).astype(int)
    proba = proba_all[:, class_index]
    print('Multiclass detected. Using one-vs-rest for class: ' + str(positive_class))
else:
    y_eval = y_test
    proba = proba_all[:,1]

if len(np.unique(y_eval)) < 2:
    raise ValueError('DecisionThreshold_Interactive requires two classes after conversion.')

tfpn = get_tfpn(y_eval, proba)

threshold = 0.5
fprtpr = get_fpr_tpr(y_eval, proba)
fprtpr.threshold = round(fprtpr.threshold, 2)
fpr_roc = fprtpr.false_positive_rate
tpr_roc = fprtpr.true_positive_rate

y_predict = model.predict(x_test)
AUC = metrics.roc_auc_score(y_eval, proba)

cms = get_cms(y_eval, proba)
cm = np.asarray(cms[int(round(threshold, 0) * 100)])
fig, ax = plt.subplots(nrows=1, ncols=2, figsize = (14, 7))
fig.tight_layout(pad = 2)
fpr, tpr = getdata(threshold, fprtpr)
coordinate = ax[0].text(fpr+0.05, tpr+0.02, s=str(fpr) + str(tpr), fontsize= 12)
graph_roc(y_eval, proba, fig, ax[0], threshold, tpr_roc, fpr_roc, AUC)
ax[1].imshow(cm, interpolation='nearest', cmap=plt.cm.Wistia)
fig.subplots_adjust(bottom=0.25)
classNames = ['Positive', 'Negative']
tick_marks = np.arange(len(classNames))
ax[1].set(ylabel = 'True label',
        xlabel = 'Predicted label',
        xticks = np.arange(len(classNames)),
        yticks = np.arange(len(classNames)),
       xticklabels = classNames,
       yticklabels = classNames)
s = [['TP', 'FN'], ['FP', 'TN']]
for i in range(2):
    for j in range(2):
        ax[1].text(j-0.3, i, str(s[i][j]) + " = " + str(cm[i][j]))
fig.subplots_adjust(bottom=0.25)

vline, hline = graph_line(ax[0], threshold, fprtpr)
point, = plot_point(ax[0], threshold, fprtpr)

sli_ax = plt.axes([0.25, 0.1, .65, .03])
thr_slider = Slider(sli_ax, 'Threshold', valmin=0, valmax=1, valinit=0.5, valstep=0.01)


def update(val):
    global point, coordinate
    ax[1].clear()
    threshold = round(thr_slider.val, 2)

    coordinate.remove()
    fpr, tpr = getdata(threshold, fprtpr)
    hline.set_ydata(y=tpr)
    vline.set_xdata(x=fpr)
    point.set_data(fpr, tpr)
    point.set_data(fpr, tpr)
    y_predict = model.predict(x_test)
    AUC = metrics.roc_auc_score(y_eval, proba)
    coordinate = ax[0].text(fpr+0.05, tpr+0.02, s=str(fpr)+str(tpr), fontsize = 12)

    cm = np.asarray(cms[int(round(threshold*100, 0))])
    ax[1].imshow(cm, interpolation='nearest', cmap=plt.cm.Wistia)
    classNames = ['Positive', 'Negative']
    tick_marks = np.arange(len(classNames))
    ax[1].set(ylabel='True label',
           xlabel='Predicted label',
           xticks=np.arange(len(classNames)),
           yticks=np.arange(len(classNames)),
           xticklabels=classNames,
           yticklabels=classNames)
    s = [['TP', 'FN'], ['FP', 'TN']]
    for i in range(2):
        for j in range(2):
            ax[1].text(j-.4, i, str(s[i][j]) + " = " + str(cm[i][j]))

    fig.canvas.draw()

thr_slider.on_changed(update)
plt.show()